In [3]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [100]:
'''topics = ['changemakers','women','agriculture','education','environment','health-care','innovation','science',
         'lifestyle','governance','sports','disability','children','entrepreneurs','waste-management-2','culture',
          'sanitation','wildlife','history','food','travel','art','water-2','pollution','youth',
          'community-development']'''
from time import sleep
from random import random, randint
def get_article_links(soup_obj):
    '''returns link for the passed soup object and throws error when the month exceeds today's month'''
    return soup_obj.find('a').get('href')

#sample link: https://www.thebetterindia.com/date/2015/3/page/2/
base_url = "https://www.thebetterindia.com/date/"
year = 2008
month = 7
page_no = 1
all_links = pd.Series()
while year < 2015:
    try:
        url = base_url+str(year)+'/'+str(month)+'/'+'page/'+str(page_no)+'/'
        resp = requests.get(url)
        soup = BeautifulSoup(resp.content,'html.parser')
    except Exception as e:
        print('Error with \n{}\nerror: {}'.format(url,e))
        pass
        
    if page_no == 1:
        try:
            pages = len(soup.find_all('nav',attrs={'class':'elementor-pagination'})[0].findChildren('a',attrs={'class':'page-numbers'}))
        except:
            pages = 1
            pass
            
    articles_in_page = pd.Series(soup.find_all('h3',attrs={'class':'elementor-post__title'}))
    all_links = pd.concat([all_links,articles_in_page.apply(get_article_links)])
    
    sleep(randint(2,9)*random())
    
    page_no += 1
    if page_no > pages:
        month+=1
        page_no=1
    if month > 12:
        year+=1
        month = 1
        
#first links after 2014 are extracted
links_after_2014 = list(all_links.values)
#then the ones upto 2014 are extracted into the same variable
links_upto_2014 = list(all_links.values)
#now upto_2014 list has all values
links_upto_2014.extend(links_after_2014)

In [97]:
#tbi's own dataframe
tbi_data = pd.DataFrame(links_upto_2014,columns=['article_link'])

In [142]:
#use it only the first time
#tbi_articles = list()    
def get_article_content(link):
    '''scrapes the actual article content from a given article_link'''
    try:
        response = requests.get(link)
        soup = BeautifulSoup(response.content,'lxml')
        paras = soup.find_all('div',attrs={'class':'entry-content clr'})[0].find_all('p')
        sleep(randint(2,9)*random())
        return ' '.join([para.get_text() for para in paras])
    except Exception as e:
        print('Error with \n{}\nerror: {}'.format(link,e))
        pass
    

tbi_articles.append(tbi_data['article_link'][5500:].apply(get_article_content))

In [146]:
articles = list()
for item in tbi_articles:
    articles.extend(item)
    
tbi_data['content'] = articles
tbi_data.to_csv('tbi_data.csv')

In [205]:
headings = list()
def get_headings(link):
    try:
        resp = requests.get(link)
        soup = BeautifulSoup(resp.content,'lxml')
        sleep(randint(2,9)*random())
        return soup.find_all('h1')[0].get_text()
    except Exception as e:
        print('Error with \n{}\nerror: {}'.format(link,e))
        pass


headings.append(tbi_data['article_link'].apply(get_headings))

Error with 
https://www.thebetterindia.com/13593/microspin-rural-innovation-cotton-farmers-become-yarn-producers-marico-siap/
error: Exceeded 30 redirects.
Error with 
https://www.thebetterindia.com/109589/a-story-of-triumph-of-hard-work-over-fate-daughter-of-a-daily-wage-labourer-enters-national-hockey-academy/
error: list index out of range
Error with 
https://www.thebetterindia.com/139182/ambalavayal-cops-transform-barren-patch-of-land-vegetable-garden/
error: list index out of range


In [210]:
headings[0].to_csv('tbi_headings.csv')

/Users/gurumanikanta/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: FutureWarning: The signature of `Series.to_csv` was aligned to that of `DataFrame.to_csv`, and argument 'header' will change its default value from False to True: please pass an explicit value to suppress this warning.
  """Entry point for launching an IPython kernel.
